# Lab 5: Categorical Predictors and Model Diagnostics

We'll use R's `iris` dataset to compare mean petal length across species while accounting for petal width. You'll fit a model, compare species with a partial F-test, and check the residuals. Two simulations then explore how the mean relationship and error variance affect the model's estimates and tests.


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 5)
source("lab5_packages.R")
if (lab5_ensure_packages()) {
    library(ottr)
    data(iris)
    cat("Rows:", nrow(iris), "\n")
    print(head(iris))
}


The `iris` data frame has measurements from 150 flowers: 50 each of `setosa`, `versicolor`, and `virginica`.

| Column | Description |
|--------|-------------|
| `Sepal.Length` | Sepal length (cm) |
| `Sepal.Width`  | Sepal width (cm) |
| `Petal.Length` | Petal length (cm) |
| `Petal.Width`  | Petal width (cm) |
| `Species`      | Species: `setosa`, `versicolor`, or `virginica` |

In this lab, use `Petal.Length` as the response and `Species` and `Petal.Width` as predictors.


## Part 1: Categorical Predictors and the Partial F-test

Fit `Petal.Length ~ Species + Petal.Width` to compare species at the same petal width. You'll interpret the fitted differences and use a partial F-test to assess the species terms.


**Question 1.1.** Which population model matches `Petal.Length ~ Species + Petal.Width` under R's default treatment coding, with `setosa` as the reference? Set `coding_model` to the number of the correct choice.

1. $\texttt{Petal.Length}_i=\beta_0+\beta_1\texttt{Species}_i+\beta_2\texttt{Petal.Width}_i+\varepsilon_i$, using the numbers 1, 2, and 3 for the three species.
2. $\texttt{Petal.Length}_i=\beta_0+\beta_1I(\texttt{Species}_i=\texttt{versicolor})+\beta_2I(\texttt{Species}_i=\texttt{virginica})+\beta_3\texttt{Petal.Width}_i+\varepsilon_i$, where each $I(\cdot)$ is a 0/1 indicator.
3. The same additive model with an intercept and a separate dummy coefficient for all three species, with no constraint on the coefficients.
4. $\texttt{Petal.Length}_i=\beta_0+\beta_1\texttt{Petal.Width}_i+\varepsilon_i$, with no species terms.


_Points:_ 0.5

In [ ]:
coding_model <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_1.R")

**Question 1.2.** Write `fit_species_models(data)` to fit and return a list with:

* `full`: the model `Petal.Length ~ Species + Petal.Width`;
* `reduced`: the model `Petal.Length ~ Petal.Width`.

Use the data frame passed to your function. The models should work with `predict()` on new data with the same column names, and your function should work on other datasets containing these columns and all three species.


_Points:_ 1

In [ ]:
fit_species_models <- function(data) {
    # YOUR CODE HERE
}


In [ ]:
species_models <- fit_species_models(iris)
summary(species_models$full)


In [ ]:
. = ottr::check("tests/q1_2.R")

**Question 1.3.** Write `species_contrast(model, species_a, species_b, width)` to return the fitted mean petal length for `species_a` minus that for `species_b`, at the same petal width, `width`. Return one number. The function should work for any pair of the model's species and when the reference species changes.

Run the comparison and plot below, then answer the interpretation question.


In [ ]:
species_contrast <- function(model, species_a, species_b, width) {
    # YOUR CODE HERE
}


In [ ]:
contrast_15 <- species_contrast(species_models$full, "virginica", "versicolor", width = 1.5)
data.frame(comparison = "virginica minus versicolor at petal width 1.5 cm",
           estimated_mean_difference_cm = contrast_15)

species_levels <- levels(iris$Species)
species_cols <- c(setosa = "steelblue", versicolor = "darkgreen", virginica = "firebrick")
plot(iris$Petal.Width, iris$Petal.Length,
     col = species_cols[as.character(iris$Species)], pch = 19,
     xlab = "Petal width (cm)", ylab = "Petal length (cm)",
     main = "Petal length and width by species")
for (species in species_levels) {
    observed_widths <- iris$Petal.Width[iris$Species == species]
    grid <- data.frame(
        Petal.Width = seq(min(observed_widths), max(observed_widths), length.out = 100),
        Species = factor(rep(species, 100), levels = species_levels))
    lines(grid$Petal.Width, predict(species_models$full, newdata = grid),
          col = species_cols[[species]], lwd = 2)
}
legend("topleft", legend = species_levels, col = species_cols[species_levels],
       pch = 19, lwd = 2, bty = "n")


Set `contrast_statements` to the numbers of all correct statements about this fitted additive model:

1. `contrast_15` estimates the mean petal-length difference, in cm, between `virginica` and `versicolor` at the same petal width of 1.5 cm.
2. The model gives the same difference at another fixed width because the species share a slope.
3. Every individual `virginica` flower must be exactly `contrast_15` cm longer than every `versicolor` flower with the same width.
4. Changing the reference species changes the coefficient labels and values; the fitted values and this contrast stay the same.
5. The contrast tells us how much petal length would change if we changed a flower's species experimentally.


_Points:_ 2

In [ ]:
contrast_statements <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_3.R")

Each line covers the petal widths observed for that species. The widths for `setosa` don't overlap with those of the other two species, so comparing `setosa` with another species at the same width requires extrapolation from the model.


**Question 1.4.** Which comparison tests whether the species terms, taken together, help explain petal length after accounting for `Petal.Width`? Set `joint_test` to one number.

1. Test whichever individual species coefficient has the smallest t-test p-value.
2. Compare the full model with an intercept-only model, testing both species coefficients and the width slope together.
3. Compare the full model with `Petal.Length ~ Petal.Width`, testing $H_0:\beta_{\texttt{versicolor}}=\beta_{\texttt{virginica}}=0$ under the default coding.
4. Compare the full model with `Petal.Length ~ Species`, testing only the width slope.


_Points:_ 1

In [ ]:
joint_test <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_4.R")

### Computing the partial F-test

Lecture 9's partial F-test compares the models' RSS values. Use `pf(F_stat, df1, df2, lower.tail = FALSE)` to get the upper-tail probability from the F distribution. The numerator degrees of freedom count the coefficients removed; the denominator uses the full model's residual degrees of freedom.


**Question 1.5.** Write `partial_f_manual(full_model, reduced_model)` to calculate
$$F=\frac{(RSS_{\mathrm{reduced}}-RSS_{\mathrm{full}})/p}{RSS_{\mathrm{full}}/(n-k)}.$$

Return a list with `n` (sample size), `k` (number of full-model coefficients, including the intercept), `p` (number of coefficients removed), `F_stat`, and the upper-tail `p_value`. Calculate the dimensions and RSS values from the models passed to your function.

Your function should handle nested, unweighted linear models fitted to the same observations, without redundant predictors. The code below compares your result for the `species_models` fits with `anova()`.


_Points:_ 2.5

In [ ]:
partial_f_manual <- function(full_model, reduced_model) {
    # YOUR CODE HERE
}


In [ ]:
f_result <- partial_f_manual(species_models$full, species_models$reduced)
anova_verification <- anova(species_models$reduced, species_models$full)
data.frame(method = c("Manual calculation", "anova()"),
           F_stat = c(f_result$F_stat, anova_verification[2, "F"]),
           p_value = c(f_result$p_value, anova_verification[2, "Pr(>F)"]))


In [ ]:
. = ottr::check("tests/q1_5.R")

**Question 1.6.** What can you conclude from your partial F-test at the 0.05 level, assuming the regression conditions hold? Set `species_conclusion` to one number.

1. There is evidence of species differences after accounting for width. The test does not tell us whether the model conditions hold.
2. Rejecting the joint null shows that every pair of species has different population mean petal lengths at fixed width.
3. A significant partial F-test establishes a causal effect of species on petal length.
4. Since the partial F-test is significant, residual diagnostics are unnecessary.


_Points:_ 1

In [ ]:
species_conclusion <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q1_6.R")

The F-test assumes the model conditions hold. Next, check this model's residuals for problems that could make its p-values unreliable.


## Part 2: Model Diagnostics and Test Behavior

Does the model from Part 1 fit the data well enough for us to trust its p-values? Start with the LINE diagnostics from Lecture 10. Then use simulations to see what happens when the mean is curved or the error variance changes.

Plots are useful to detect any potential problems, but they cannot confirm every model assumption. The `iris` rows are grouped by species, with no collection order recorded. Their row numbers therefore cannot establish independence between observations.


**Question 2.1.** Write `diagnostic_summary(model, groups)` to return a list with:

* `residuals` and `fitted`: the model's residuals and fitted values;
* `shapiro_p`: the Shapiro-Wilk p-value for the residuals;
* `group_sd`: residual standard deviations for each group, named by group.

`groups` contains one label for each observation used to fit the model. Use the model and labels passed to your function. The cells below call it on the Part 1 model and draw the diagnostic plots.


_Points:_ 2

In [ ]:
diagnostic_summary <- function(model, groups) {
    # YOUR CODE HERE
}

In [ ]:
d_result <- diagnostic_summary(species_models$full, iris$Species)

In [ ]:
. = ottr::check("tests/q2_1.R")

Look for curves and changes in spread in the `iris` residuals. Use the histogram, Q-Q plot, and Shapiro-Wilk result together as checks on their shape. These checks cannot establish how the unobserved model errors are distributed. Pooling groups with different variances can look non-Normal overall even if errors within each group are Normal.


In [ ]:
local({
    old_par <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
    on.exit(par(old_par))
    species_cols <- c(setosa = "#406A8E", versicolor = "#3F7B59", virginica = "#B35B36")
    plot(d_result$fitted, d_result$residuals, pch = 19,
         col = adjustcolor(species_cols[as.character(iris$Species)], alpha.f = 0.7),
         xlab = "Fitted petal length (cm)", ylab = "Residual (cm)",
         main = "Residuals vs fitted")
    abline(h = 0, lty = 2)
    boxplot(d_result$residuals ~ iris$Species, col = unname(species_cols),
            xlab = "Species", ylab = "Residual (cm)", main = "Residual spread by species")
    abline(h = 0, lty = 2)
})

In [ ]:
local({
    old_par <- par(mfrow = c(1, 2), mar = c(4, 4, 2, 1))
    on.exit(par(old_par))
    hist(d_result$residuals, breaks = 15, col = "#B7CBD9", border = "white",
         main = "Residual histogram", xlab = "Residual (cm)")
    qqnorm(d_result$residuals, pch = 19, col = "#406A8E", main = "Normal Q-Q plot")
    qqline(d_result$residuals, lwd = 2, col = "#B35B36")
})
print(data.frame(Species = names(d_result$group_sd),
                 Residual_SD = as.numeric(d_result$group_sd)), row.names = FALSE)
d_result$shapiro_p

**Question 2.2.** Use the `iris` output above to set:

* `largest_spread_species`: the name of the species with the largest value in `d_result$group_sd`. Include all names if there is a tie.
* `normality_reject`: your Shapiro-Wilk decision at the 0.05 level (`TRUE` to reject).


_Points:_ 1

In [ ]:
largest_spread_species <- NULL # YOUR CODE HERE
normality_reject <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_2.R")

### What remains in the residuals?

This example uses independent observations from
$$x_i\sim\operatorname{Uniform}(-5,5),\qquad
y_i=0.5x_i^2+\varepsilon_i,\qquad \varepsilon_i\sim N(0,5^2),$$
with errors independent of the predictors. The code below fits a straight-line model and a model with both $x$ and $x^2$ to the same dataset.

Compare their residuals against $x$. Both panels use the same vertical scale, and the dashed lines mark zero.


In [ ]:
set.seed(52710)
curved_x <- runif(100, -5, 5)
curved_y <- 0.5 * curved_x^2 + rnorm(100, sd = 5)
curved_line_fit <- lm(curved_y ~ curved_x)
curved_quad_fit <- lm(curved_y ~ curved_x + I(curved_x^2))
curved_residuals <- list(residuals(curved_line_fit), residuals(curved_quad_fit))
curved_limits <- range(unlist(curved_residuals))
curved_par <- par(mfrow = c(1, 2), mar = c(4.5, 4.5, 3, 0.8))
for (panel in seq_along(curved_residuals)) {
    plot(curved_x, curved_residuals[[panel]], pch = 19,
         col = adjustcolor("#406A8E", 0.65),
         xlab = "Predictor x", ylab = "Residual", ylim = curved_limits,
         main = c("A: Straight-line fit", "B: Quadratic fit")[panel])
    abline(h = 0, lty = 2, col = "gray40")
}
par(curved_par)


**Question 2.3.** Which statements are supported by these residual plots? Set `curvature_statements` to the numbers of all correct statements.

1. The U-shaped pattern in Panel A suggests that the straight-line model misses part of the mean relationship.
2. Because the residuals average to zero in Panel A, its mean model must be correct.
3. Panel B proves that the model errors are independent, Normal, and have equal variance.
4. The clear U-shaped residual trend in Panel A is no longer visible in Panel B after including $x^2$.


_Points:_ 1

In [ ]:
curvature_statements <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_3.R")

**Question 2.4.** Write `sim_mean_tests(n, beta1, beta2, sigma, reps)` to estimate rejection rates for data from
$$x_i\sim\operatorname{Uniform}(-5,5),\qquad
y_i=\beta_1x_i+\beta_2x_i^2+\varepsilon_i,\qquad
\varepsilon_i\sim N(0,\sigma^2).$$
The observations are independent, and the errors are independent of the predictors.

Return a list with two rejection rates at the 0.05 level:

* `linear_rate`: the two-sided slope t-test in `lm(y ~ x)`;
* `quadratic_rate`: the joint F-test comparing `lm(y ~ x + I(x^2))` with `lm(y ~ 1)`.

`I(x^2)` adds the squared predictor to the model. Use both tests on the same new dataset in each replicate. Your function should use all five arguments.


In [ ]:
sim_mean_tests <- function(n, beta1, beta2, sigma, reps) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(52741)
mean_test_rates <- sim_mean_tests(n = 50, beta1 = 0, beta2 = 0.5, sigma = 5, reps = 1000)
mean_rate_values <- c(mean_test_rates$linear_rate, mean_test_rates$quadratic_rate)
mean_rate_bars <- barplot(mean_rate_values,
    names.arg = c("Straight-line\nslope t-test", "Joint F-test\n(x and x^2)"),
    col = c("#B35B36", "#406A8E"),
    ylab = "Estimated rejection rate at alpha = 0.05",
    main = "Testing a curved mean", ylim = c(0, 1.15))
text(mean_rate_bars, mean_rate_values + 0.03, labels = sprintf("%.3f", mean_rate_values))
abline(h = 0.05, lty = 2)


Compare the rejection-rate plot with the residual plots in Q2.3. Set `mean_test_statements` to the numbers of all correct statements.

1. A non-significant straight-line slope rules out the curved mean pattern seen in Panel A.
2. Including $x^2$ addresses the mean pattern left by the straight-line fit, and the joint F-test can detect this curved relationship.
3. The two bars estimate power for the same null hypothesis because both models use the same observations.
4. A zero population straight-line slope does not guarantee a 5% false-positive rate for the usual t-test when the mean is curved.


_Points:_ 3.5

In [ ]:
mean_test_statements <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_4.R")

The residual plots show what the straight-line model misses. In this population, symmetry gives a zero straight-line slope even though the mean changes with $x$. Including $x^2$ captures that curve; a non-significant straight-line slope alone does not rule out an association.


### Unequal variance: slope estimates and p-values

For this simulation, use
$$x_i\sim\operatorname{Uniform}(-5,5),\qquad y_i=\beta_1x_i+\varepsilon_i.$$
The observations are independent, with $E(\varepsilon_i\mid x_i)=0$. The two models have the same mean but different error spreads: a constant SD of 5, or an SD of $0.5+2|x_i|$.

**Question 2.5.** Write `sim_variance_tests(n, reps, beta1 = 0, unequal = FALSE)`. Use Normal errors with constant SD when `unequal = FALSE` and the varying SD above when `unequal = TRUE`. Generate a new dataset in each replicate, fit `lm(y ~ x)`, and return a list with:

* `mean_slope`: the average fitted slope across replicates;
* `rejection_rate`: the proportion of two-sided slope t-tests with p-value below 0.05.

Your function should work with zero and nonzero slopes. The experiment below uses zero slopes, so its rejection rates estimate Type I error.


In [ ]:
sim_variance_tests <- function(n, reps, beta1 = 0, unequal = FALSE) {
    # YOUR CODE HERE
}


In [ ]:
set.seed(52751)
variance_constant <- sim_variance_tests(n = 50, reps = 2000, beta1 = 0, unequal = FALSE)
set.seed(52751)
variance_unequal <- sim_variance_tests(n = 50, reps = 2000, beta1 = 0, unequal = TRUE)
variance_results <- list(constant = variance_constant, unequal = variance_unequal)
variance_table <- data.frame(
    error_spread = c("Constant SD", "SD increases with |x|"),
    true_slope = c(0, 0),
    mean_slope = c(variance_results$constant$mean_slope, variance_results$unequal$mean_slope),
    rejection_rate = c(variance_results$constant$rejection_rate, variance_results$unequal$rejection_rate))
print(variance_table, row.names = FALSE)
variance_bars <- barplot(variance_table$rejection_rate,
    names.arg = c("Constant SD", "SD increases\nwith |x|"),
    col = c("#406A8E", "#B35B36"), ylab = "Estimated Type I error rate",
    main = "False positives under two error-variance models",
    ylim = c(0, max(variance_table$rejection_rate, 0.05) * 1.4))
text(variance_bars, variance_table$rejection_rate + 0.005,
     labels = sprintf("%.3f", variance_table$rejection_rate))
abline(h = 0.05, lty = 2)


Compare the average slopes and rejection rates in the table. Set `variance_statements` to the numbers of all correct statements.

1. In these models, changing the error variance alone does not change the true slope or make the OLS slope estimator biased.
2. An average fitted slope near zero guarantees a 5% false-positive rate for the usual slope test.
3. Every pattern of unequal error variance must inflate Type I error by the same amount.
4. These results show that fitted slopes can average near the true value even when the test rejects too often.


_Points:_ 3.5

In [ ]:
variance_statements <- NULL # YOUR CODE HERE


In [ ]:
. = ottr::check("tests/q2_5.R")

Here, errors are larger at the extremes of $x$, where observations have more influence on the fitted slope. The usual standard error assumes constant variance and can be too small here, even though the slope estimator is unbiased. This explains the extra false positives for this particular variance pattern.


### Back to the iris data

Return to your species comparison. Use the fitted mean difference, partial F-test, and residual diagnostics to answer the final question.


In [ ]:
c(virginica_minus_versicolor_cm = contrast_15,
  species_F = f_result$F_stat,
  species_p = f_result$p_value)

**Question 2.6.** Which statements belong in a summary of your analysis? Set `final_statements` to the numbers of all correct statements.

1. A small partial F-test p-value shows that changing a flower's species would cause its petal length to change.
2. `contrast_15` estimates the difference in **mean** petal length, in cm, for virginica minus versicolor at a petal width of 1.5 cm. It does not say that every pair of individual flowers differs by that amount.
3. The partial F-test tests the species terms together after accounting for width. Differences in residual spread across species raise doubts about the equal-variance assumption used for its usual p-value.
4. With 150 observations, unequal error variance no longer matters for the usual F-test.
5. A joint species F-test and an individual species-coefficient t-test test the same null hypothesis.
6. In the unequal-variance simulation, fitted slopes can average near the true slope while the usual test produces too many false positives.


_Points:_ 1

In [ ]:
final_statements <- NULL # YOUR CODE HERE

In [ ]:
. = ottr::check("tests/q2_6.R")

## Submission

1. Restart the kernel and run all cells from top to bottom. In Jupyter, use `Kernel -> Restart Kernel and Run All Cells...` or the equivalent command in your interface.
2. Fix any errors and check that every grader check passes. If something fails, start with the earliest error, then rerun the notebook.
3. Save the notebook, download the `.ipynb` file, and upload it to Gradescope.
4. Wait for grading to finish and check that all tests pass on Gradescope. Keep your submission confirmation.
